In [1]:
import os
import json
import re
import requests
from pathlib import Path


In [2]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

True

In [3]:
ALPHAVANTAGE_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print(ALPHAVANTAGE_KEY)

PFP2SVUSBI040YNE


In [4]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [5]:
PROCESSED_DIR = Path("../data/processed")

# Test on AAPL Q1 2023
test_file = PROCESSED_DIR / "AAPL_Q1_2023.json"

with open(test_file, encoding="utf-8") as f:
    transcript = json.load(f)

print("Company:", transcript["symbol"])
print("Quarter:", transcript["quarter"])
print("Year:", transcript["year"])
print("Forward guidance sentences found:", len(transcript["forward_guidance"]))
print()
print("First 3 guidance sentences:")
for s in transcript["forward_guidance"][:3]:
    print(" -", s[:150])

Company: AAPL
Quarter: 1
Year: 2023
Forward guidance sentences found: 11

First 3 guidance sentences:
 - Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limita
 - As we move into the March quarter, I'd like to review our outlook, which includes the types of forward-looking information that Tejas referred to at t
 - Given the continued uncertainty around the world in the near term, we are not providing revenue guidance, but we are sharing some directional insights


In [6]:
def extract_guidance(guidance_sentences, symbol, quarter, year):
    if not guidance_sentences:
        return []

    sentences_text = "\n".join( f"{i+1}. {s}" for i, s in enumerate(guidance_sentences))

    prompt = f"""You are a financial analyst extracting forward guidance from an earnings call transcript.

Company: {symbol}
Quarter: Q{quarter} {year}

Below are sentences that may contain forward guidance (management forecasts about future performance).
For each sentence that contains REAL forward guidance (actual numerical or directional forecasts), extract structured data.
Skip sentences that are just disclaimers, boilerplate, or vague statements with no specific forecast.

Sentences:
{sentences_text}

Return a JSON array. Each item must have exactly these fields:
{{
  "metric": "Revenue" or "EPS" or "Margin" or "CapEx" or "Growth" or "Volume" or "Other",
  "value": "exact value or range mentioned e.g. 8-10% or high single digits or flat or $2.3B",
  "direction": "increase" or "decrease" or "flat" or "range" or "unknown",
  "time_period": "Next Quarter" or "Full Year" or "FY{year+1}" or "Q{quarter+1} {year}" or exact period mentioned,
  "raw_sentence": "original sentence"
}}

Return ONLY the JSON array. If no real guidance found, return empty array [].
No other text."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system",
             "content": "You are a strict financial analyst. Follow the instruction to find out genuine future guidance."
            },
            {"role": "user",
             "content": prompt
            }
        ]
    )

    text = response.choices[0].message.content.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        result = json.loads(text)
        return result if isinstance(result, list) else []
    except Exception as e:
        print("JSON Error:", e)
        print(text)
        return []


In [9]:
# Test on our sample transcript
guidance_extracted = extract_guidance(
    transcript["forward_guidance"],
    transcript["symbol"],
    transcript["quarter"],
    transcript["year"]
)

print(f"Guidance claims extracted: {len(guidance_extracted)}")
print()
for item in guidance_extracted:
    print(f"Metric: {item['metric']}")
    print(f"Value: {item['value']}")
    print(f"Direction: {item['direction']}")
    print(f"Time Period: {item['time_period']}")
    print(f"Sentence: {item['raw_sentence'][:100]}")
    print()

Guidance claims extracted: 7

Metric: Revenue
Value: similar to the December quarter
Direction: flat
Time Period: March quarter
Sentence: In total, we expect our March quarter year-over-year revenue performance to be similar to the Decemb

Metric: Revenue
Value: 5 percentage points
Direction: decrease
Time Period: Next Quarter
Sentence: Foreign exchange will continue to be a headwind, and we expect a negative year-over-year impact of 5

Metric: Revenue
Value: accelerate
Direction: increase
Time Period: March quarter
Sentence: For iPhone, we expect our March quarter year-over-year revenue performance to accelerate relative to

Metric: Revenue
Value: double digits
Direction: decrease
Time Period: March quarter
Sentence: For Mac and iPad, we expect revenue for both product categories to decline double digits year-over-y

Metric: Gross Margin
Value: 43.5-44.5%
Direction: range
Time Period: Next Quarter
Sentence: We expect gross margin to be between 43.5% and 44.5%

Metric: OpEx
Value: $13.

In [7]:
def fetch_quarter_data(symbol, quarter, year):
    """
    Fetch Wall Street EPS estimate vs actual reported EPS
    for a specific company and quarter from Alpha Vantage.
    Returns dict with estimate, actual, surprise, revenue data.
    """
    url = "https://www.alphavantage.co/query"

    # Fetch earnings history
    params = {
        "function": "EARNINGS",
        "symbol": symbol,
        "apikey": ALPHAVANTAGE_KEY
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
    except Exception as e:
        print(f"API error: {e}")
        return None

    quarterly_earnings = data.get("quarterlyEarnings", [])
    if not quarterly_earnings:
        return None

    # Find the specific quarter we want
    # Alpha Vantage fiscal date: match by year and approximate quarter
    target_entry = None
    for entry in quarterly_earnings:
        date = entry.get("fiscalDateEnding", "")
        if not date or len(date) < 7:
            continue
        entry_year  = int(date[:4])
        entry_month = int(date[5:7])
        entry_quarter = (entry_month - 1) // 3 + 1

        if entry_year == year and entry_quarter == quarter:
            target_entry = entry
            break

    if not target_entry:
        print(f"No data found for {symbol} Q{quarter} {year}")
        return None

    try:
        reported_eps  = float(target_entry.get("reportedEPS") or 0)
        estimated_eps = float(target_entry.get("estimatedEPS") or 0)
        surprise_pct  = float(target_entry.get("surprisePercentage") or 0)
    except (ValueError, TypeError):
        reported_eps  = 0.0
        estimated_eps = 0.0
        surprise_pct  = 0.0

    return {
        "symbol": symbol,
        "quarter": quarter,
        "year": year,
        "reported_eps": reported_eps,
        "estimated_eps": estimated_eps,
        "surprise_pct": round(surprise_pct, 2),
        "beat_estimate": reported_eps > estimated_eps,
        "fiscal_date": target_entry.get("fiscalDateEnding", "")
    }


In [ ]:
# Test fetch for AAPL Q1 2023
actuals = fetch_quarter_data("AAPL", 1, 2023)
print("Wall Street vs Actual for AAPL Q1 2023:")
print(json.dumps(actuals, indent=2))

In [8]:
def fetch_revenue_actuals(symbol, quarter, year):
    """
    Fetch actual revenue for a specific quarter from Alpha Vantage
    income statement data.
    """
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "INCOME_STATEMENT",
        "symbol":   symbol,
        "apikey":   ALPHAVANTAGE_KEY
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
    except Exception as e:
        print(f"API error: {e}")
        return None

    quarterly_reports = data.get("quarterlyReports", [])
    if not quarterly_reports:
        return None

    for entry in quarterly_reports:
        date = entry.get("fiscalDateEnding", "")
        if not date or len(date) < 7:
            continue
        entry_year = int(date[:4])
        entry_month = int(date[5:7])
        entry_quarter = (entry_month - 1) // 3 + 1

        if entry_year == year and entry_quarter == quarter:
            try:
                revenue = float(entry.get("totalRevenue") or 0)
                net_income = float(entry.get("netIncome") or 0)
                gross_profit = float(entry.get("grossProfit") or 0)
                gross_margin = round(gross_profit / revenue * 100, 2) if revenue > 0 else 0
            except (ValueError, TypeError):
                revenue = net_income = gross_margin = 0.0

            return {
                "symbol": symbol,
                "quarter": quarter,
                "year": year,
                "revenue": revenue,
                "net_income": net_income,
                "gross_margin": gross_margin,
                "fiscal_date": date
            }

    print(f"No revenue data found for {symbol} Q{quarter} {year}")
    return None



In [ ]:
# Test
revenue_data = fetch_revenue_actuals("AAPL", 1, 2023)
print("Revenue actuals for AAPL Q1 2023:")
print(json.dumps(revenue_data, indent=2))

In [ ]:
def compare_guidance_vs_actuals(guidance_list, eps_actuals, revenue_actuals):
    """
    Compare each extracted guidance claim against actual reported results.
    Returns a structured comparison with verdict: BEAT / MET / MISSED / UNVERIFIABLE
    """
    if not guidance_list:
        return []

    comparisons = []

    for claim in guidance_list:
        metric = claim["metric"]
        value = claim["value"]
        direction = claim["direction"]

        verdict = "UNVERIFIABLE"
        actual_val = None
        explanation = ""

        # EPS comparison
        if metric == "EPS" and eps_actuals:
            reported  = eps_actuals["reported_eps"]
            estimated = eps_actuals["estimated_eps"]
            actual_val = reported

            if eps_actuals["beat_estimate"]:
                verdict = "BEAT"
                explanation = f"Reported EPS {reported} beat estimate {estimated}"
            elif abs(reported - estimated) < 0.02:
                verdict = "MET"
                explanation = f"Reported EPS {reported} roughly met estimate {estimated}"
            else:
                verdict = "MISSED"
                explanation = f"Reported EPS {reported} missed estimate {estimated}"

        # Revenue comparison
        elif metric == "Revenue" and revenue_actuals:
            actual_val = revenue_actuals["revenue"]

            if direction == "increase" and actual_val > 0:
                verdict = "BEAT" if eps_actuals and eps_actuals["beat_estimate"] else "MET"
                explanation = f"Revenue reported: ${actual_val:,.0f}"
            elif direction == "decrease":
                verdict = "MET"
                explanation = f"Revenue reported: ${actual_val:,.0f}"
            else:
                verdict = "UNVERIFIABLE"
                explanation = "Cannot precisely verify directional revenue claim"

        # Margin comparison
        elif metric == "Margin" and revenue_actuals:
            actual_val = revenue_actuals["gross_margin"]
            explanation = f"Gross margin reported: {actual_val}%"

            if direction == "increase":
                verdict = "UNVERIFIABLE"
                explanation += " (need prior quarter to verify direction)"
            else:
                verdict = "UNVERIFIABLE"

        comparisons.append({
            "metric": metric,
            "guidance": value,
            "direction": direction,
            "time_period": claim["time_period"],
            "actual": actual_val,
            "verdict": verdict,
            "explanation": explanation,
            "raw_sentence": claim["raw_sentence"]
        })

    return comparisons



In [ ]:
# Test full comparison
comparisons = compare_guidance_vs_actuals(
    guidance_extracted,
    actuals,
    revenue_data
)

print(f"Total guidance claims compared: {len(comparisons)}")
print()
for c in comparisons:
    print(f"Metric:     {c['metric']}")
    print(f"Guidance:   {c['guidance']} ({c['direction']}) for {c['time_period']}")
    print(f"Actual:     {c['actual']}")
    print(f"Verdict:    {c['verdict']}")
    print(f"Explanation:{c['explanation']}")
    print()

In [ ]:
def build_guidance_report(symbol, quarter, year, comparisons, eps_actuals, revenue_actuals):
    """
    Build the complete structured guidance report for one transcript.
    This is what gets returned to the frontend dashboard.
    """
    total     = len(comparisons)
    beat      = sum(1 for c in comparisons if c["verdict"] == "BEAT")
    met       = sum(1 for c in comparisons if c["verdict"] == "MET")
    missed    = sum(1 for c in comparisons if c["verdict"] == "MISSED")
    unverifiable = sum(1 for c in comparisons if c["verdict"] == "UNVERIFIABLE")

    # Credibility score: beat=1.0, met=0.5, missed=0.0, unverifiable ignored
    verifiable = beat + met + missed
    if verifiable > 0:
        credibility_score = round((beat * 1.0 + met * 0.5) / verifiable * 100)
    else:
        credibility_score = None

    report = {
        "symbol":   symbol,
        "quarter":  quarter,
        "year":     year,
        "summary": {
            "total_guidance_claims": total,
            "beat":         beat,
            "met":          met,
            "missed":       missed,
            "unverifiable": unverifiable,
            "credibility_score": credibility_score
        },
        "eps_actuals":     eps_actuals,
        "revenue_actuals": revenue_actuals,
        "detailed_comparisons": comparisons
    }

    return report



In [ ]:
# Build and display the report
report = build_guidance_report(
    transcript["symbol"],
    transcript["quarter"],
    transcript["year"],
    comparisons,
    actuals,
    revenue_data
)

print("=" * 60)
print(f"GUIDANCE REPORT: {report['symbol']} Q{report['quarter']} {report['year']}")
print("=" * 60)
print()
print("SUMMARY:")
print(f"  Total guidance claims: {report['summary']['total_guidance_claims']}")
print(f"  Beat:          {report['summary']['beat']}")
print(f"  Met:           {report['summary']['met']}")
print(f"  Missed:        {report['summary']['missed']}")
print(f"  Unverifiable:  {report['summary']['unverifiable']}")
print(f"  Credibility Score: {report['summary']['credibility_score']}/100")
print()
print("DETAILED:")
for c in report["detailed_comparisons"]:
    print(f"  [{c['verdict']}] {c['metric']}: {c['guidance']} → Actual: {c['actual']}")

In [ ]:
OUTPUT_DIR = Path("../data/guidance_reports")
OUTPUT_DIR.mkdir(exist_ok=True)

output_path = OUTPUT_DIR / f"{report['symbol']}_Q{report['quarter']}_{report['year']}_guidance.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Report saved to:", output_path)

In [ ]:
import time

In [ ]:
all_reports = []
failed_files = []
processed_dir = Path("../data/processed")
output_dir = Path("../data/guidance_reports")
output_dir.mkdir(exist_ok=True)

all_files = [
    f for f in processed_dir.glob("*.json")
    if not f.name.startswith("_")
]

print(f"Processing {len(all_files)} transcripts...")
print()

for i, filepath in enumerate(all_files):
    output_path = output_dir / filepath.name.replace(".json", "_guidance.json")

    if output_path.exists():
        print(f"[{i+1}/{len(all_files)}] {filepath.name} - Already done, skipping")
        continue

    try:
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)

        symbol  = data["symbol"]
        quarter = data["quarter"]
        year    = data["year"]

        # Phase 1-5: Extract guidance from transcript
        guidance = extract_guidance(
            data["forward_guidance"], symbol, quarter, year
        )

        # Phase 6: Fetch actuals from Alpha Vantage
        eps_data = fetch_quarter_actuals(symbol, quarter, year)
        time.sleep(12)  # Alpha Vantage rate limit: 5 calls/min

        rev_data = fetch_revenue_actuals(symbol, quarter, year)
        time.sleep(12)

        # Compare
        comparisons = compare_guidance_vs_actuals(guidance, eps_data, rev_data)

        # Build report
        report = build_guidance_report(
            symbol, quarter, year, comparisons, eps_data, rev_data
        )

        # Save
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2)

        score = report["summary"]["credibility_score"]
        print(f"[{i+1}/{len(all_files)}] {filepath.name} - "
              f"{len(guidance)} claims | Score: {score}")

    except Exception as e:
        print(f"[{i+1}/{len(all_files)}] {filepath.name} - FAILED: {e}")
        failed_files.append(filepath.name)

print()
print("Done.")
print("Failed files:", len(failed_files))
if failed_files:
    for f in failed_files:
        print(" ", f)